# 2-3 기사 유형 분류 (키워드 규칙 v1)
1. 설정 → 2. 규칙 불러오기 → 3. 기사 하나 시험 → 4. 분류시험 100건 일치율 → 5. 전체 기사 분류

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/4팀_분류시험'   # 정답지가 있는 폴더
import os, pandas as pd, numpy as np; os.makedirs(OUT, exist_ok=True)

## 규칙 (키워드는 `KW` 목록에서 넣고 빼기)

In [ ]:
"""2-3 기사 유형 분류: 키워드 규칙 v1 (초안)

판단 근거: 제목 + 본문 앞 200자 (팀 분류 지시문과 같음). 주가·수익률·전망 정보는 판단에 쓰지 않는다.
순서: [기타-시황/리포트 먼저 거름] → 광고 → 공시형 → 논란형 → 화제형 → 기타
복합: 두 유형 이상 키워드가 걸리면 앞 순서로 정하고 복합=Y.
      단, 공시형 + 논란형이 같이 걸리면 논란형 (교수님 예: "유상증자가 논란" → 논란형).
제목에서 먼저 찾고, 제목에 아무것도 없을 때만 본문 앞 200자에서 찾는다.
"""
import re

# 주가·시황·리포트·칼럼: 수익률 정보가 섞이므로 '기타'로 먼저 거른다
OTHER_FIRST = [
    "목표주가", "목표가", "투자의견", "매수 유지", "매수의견", "주가", "시황", "증시", "코스피", "코스닥",
    "특징주", "종목", "펀드", "ETF", "배당주", "수익률", "상한가", "하한가", "신고가", "급등", "급락", "약세", "강세",
    "칼럼", "기고", "사설", "[마켓인사이트]", "HOT종목", "주간", "전망",
    "배당주", "고배당", "투자스토리", "개미", "기관 외인", "株", "보수 1위", "사볼까", "재테크",
]
# 제목이 이 모양이면 증권사 리포트 → 기타  예: "KT&G, ... 시동"-하이투자
REPORT = re.compile(r'"\s*-\s*\S+(증권|투자)\s*$|-\s*\S+증권\s*$')
KW = {
    "광고": [
        "[광고]", "[AD]", "[PR]", "협찬", "기획광고",
        "사회공헌", "기부", "후원", "봉사", "장학", "상생", "기금", "출연", "성금", "나눔", "지원금", "복지",
        "수상", "표창", "시상", "대상 수상", "인증 획득", "인증을 획득", "선정돼", "으뜸기업",
        "공모전", "모집", "채용", "캠페인", "페스티벌", "콘서트", "전시", "공연", "메세나", "문화예술",
        "ESG 경영", "탄소중립", "친환경 활동", "플로깅", "임직원",
        "브랜드 대상", "브랜드경쟁력", "경쟁력지수", "올해의 브랜드", "우수기업", "인증", "소비자 선정", "광고대상",
    ],
    "공시형": [
        "공시", "잠정", "실적", "영업이익", "영업익", "매출", "순이익", "순익", "흑자", "적자", "분기",
        "배당", "자사주", "주주환원", "유상증자", "무상증자", "증자", "감자", "회사채", "채권 발행", "전환사채", "CB 발행", "BW",
        "수주", "공급계약", "단일판매", "계약 체결", "주총", "주주총회", "이사회", "사장 후보", "대표이사 선임", "선임",
        "영업정지", "판매 중단", "사업 중단", "액면분할", "신규 상장", "상장폐지", "합병 결정", "분할 결정",
    ],
    "논란형": [
        "횡령", "배임", "산재", "불매", "소송", "피소", "환경오염", "오염", "압수수색", "검찰", "경찰", "수사", "구속",
        "영장", "기소", "의혹", "과징금", "제재", "적발", "리콜", "결함", "파업", "노조", "갑질", "담합", "유출",
        "사고", "화재", "폭발", "사망", "논란", "고발", "징계", "벌금", "공정위", "부당", "불공정", "비자금", "분식", "탈세",
    ],
    "화제형": [
        "출시", "신제품", "신사업", "론칭", "런칭", "선보", "제휴", "협업", "협약", "MOU", "업무협약", "파트너",
        "진출", "수출", "해외", "법인 설립", "개발", "특허", "기술", "인수", "매각", "합작", "투자 유치", "지분 투자", "설비 투자", "컨소시엄",
        "우선협상", "사업자 선정", "착공", "준공", "가동", "점유율", "돌파", "최초", "확대",
    ],
}
ORDER = ["광고", "공시형", "논란형", "화제형"]


def _hits(text):
    return {t: [k for k in KW[t] if k in text] for t in ORDER}


def classify(title, body=""):
    """반환: (type, 복합 'Y'/'N', 근거 키워드)"""
    title = str(title or ""); body = str(body or "")[:200]
    if REPORT.search(title):
        return "기타", "N", "증권사 리포트"
    strong = ["배당주", "고배당", "펀드", "투자스토리", "개미", "기관 외인", "株", "사볼까", "칼럼"]
    if any(k in title for k in strong):
        return "기타", "N", next(k for k in strong if k in title)
    if any(k in title for k in OTHER_FIRST) and not any(k in title for k in KW["논란형"] + ["공시", "잠정", "실적", "배당", "자사주", "판매 중단", "영업정지"]):
        return "기타", "N", next(k for k in OTHER_FIRST if k in title)
    h = _hits(title)
    if not any(h.values()):
        h = _hits(body)
    got = [t for t in ORDER if h[t]]
    if not got:
        return "기타", "N", ""
    multi = "Y" if len(got) > 1 else "N"
    t = got[0]
    if "공시형" in got and "논란형" in got:      # 공시 사실이 논란의 대상 → 논란형
        t = "논란형"
    if t == "광고" and "논란형" in got:           # 홍보성 문구가 있어도 사건이면 논란형
        t = "논란형"
    return t, multi, h[t][0]


## 기사 하나 시험

In [ ]:
print(classify('삼성전자, 3분기 영업이익 10조 돌파', ''))
print(classify('OO기업 회장 횡령 혐의 압수수색', ''))
print(classify('LG전자, 세계 최초 롤러블 TV 출시', ''))

## 분류시험 100건 일치율 (예원·아영 정답을 받은 뒤)
사람 둘이 다르게 단 기사는 `분류시험_100_최종정답.xlsx`의 '최종' 칸을 채우고 다시 실행.

In [ ]:
VALID = ["광고", "공시형", "논란형", "화제형", "기타"]


def _norm(v):
    v = str(v).strip().replace(" ", "")
    for t in VALID:
        if v.startswith(t[:2]):
            return t
    return np.nan


def agreement(path_a, path_b, classify, out_dir, final_col=None):
    """[3] 사람1·사람2·컴퓨터 비교. 사람 둘이 다른 기사는 '최종' 칸을 영섭이 채우고 다시 실행"""
    col = "정답 (광고/공시형/논란형/화제형/기타)"
    a = pd.read_excel(path_a, dtype=str); b = pd.read_excel(path_b, dtype=str)
    m = a[["번호", "id", "회사", "제목", "본문 앞 200자", col, "복합 (Y/N)"]].merge(
        b[["번호", col, "복합 (Y/N)"]], on="번호", suffixes=("_A", "_B"))
    m["사람A"] = m[f"{col}_A"].map(_norm); m["사람B"] = m[f"{col}_B"].map(_norm)
    hh = (m.사람A == m.사람B).mean()
    m["최종"] = np.where(m.사람A == m.사람B, m.사람A, np.nan)
    fp = f"{out_dir}/분류시험_100_최종정답.xlsx"
    if os.path.exists(fp):                      # 영섭이 채운 최종 정답 반영
        f = pd.read_excel(fp, dtype=str).set_index("번호")["최종"].map(_norm)
        m["최종"] = m.번호.map(f).fillna(m["최종"])
    r = m.apply(lambda x: classify(x["제목"], x["본문 앞 200자"]), axis=1, result_type="expand")
    m[["컴퓨터", "컴퓨터_복합", "근거"]] = r
    ok = m.최종.notna()
    acc = (m.loc[ok, "컴퓨터"] == m.loc[ok, "최종"]).mean()
    tp = m[ok & (m.컴퓨터 == "화제형")]; tr = m[ok & (m.최종 == "화제형")]
    prec = (tp.최종 == "화제형").mean() if len(tp) else np.nan
    rec = (tr.컴퓨터 == "화제형").mean() if len(tr) else np.nan
    print(f"사람끼리 일치 {hh:.0%} ({(m.사람A == m.사람B).sum()}/{len(m)})")
    print(f"최종 정답이 정해진 기사 {ok.sum()}건 | 컴퓨터 일치율 {acc:.0%} (목표 80%) | 화제형 정밀도 {prec:.0%}, 재현율 {rec:.0%}")
    print(pd.crosstab(m.loc[ok, "최종"], m.loc[ok, "컴퓨터"], rownames=["사람 최종"], colnames=["컴퓨터"], margins=True))
    if not os.path.exists(fp):
        m[["번호", "회사", "제목", "사람A", "사람B", "최종"]].to_excel(fp, index=False)
        print(f"→ 사람 둘이 다른 {(~ok).sum()}건: {fp} 의 '최종' 칸을 채우고 이 셀을 다시 실행")
    m[ok & (m.컴퓨터 != m.최종)][["번호", "회사", "제목", "최종", "컴퓨터", "근거"]].to_excel(f"{out_dir}/분류시험_100_틀린것.xlsx", index=False)
    return m




In [ ]:
m = agreement(f'{OUT}/분류시험_100_예원.xlsx', f'{OUT}/분류시험_100_아영.xlsx', classify, OUT)

## 전체 기사 분류 (articles_clean 파일)

In [ ]:
CLEAN = '/content/drive/MyDrive/뉴스_원문_비공개/articles_clean_200.csv'   # ← 경로 바꾸기
x = pd.read_csv(CLEAN, dtype=str)
x[['type','복합','근거']] = x.apply(lambda r: classify(str(r.get('title','')), str(r.get('body',''))), axis=1, result_type='expand')
print(x.type.value_counts(normalize=True).round(3))
x.to_csv(CLEAN.replace('.csv','_typed.csv'), index=False, encoding='utf-8-sig')   # 원문 포함 → 비공개 드라이브에만